# Proyecto Final: Pipeline Integral de Limpieza y Análisis Exploratorio de Datos (EDA)

**Tema:** Ventas y satisfacción en el marketplace brasileño Olist (2016-2018)

## Datos académicos

- **Materia:** Ciencia de Datos
- **Docente:** Fernando Mayorga Guittins
- **Fecha:** octubre de 2026
- **Integrantes del equipo:**
  - Arturo Obed Velázquez Téllez
  - Ángel de Jesús González Molina
  - Antonia Martínez Santos
  - José Carlos Hernández Ballado

## Contexto de los datos

- **Fuente:** *Brazilian E-Commerce Public Dataset by Olist* (Kaggle, `olistbr/brazilian-ecommerce`).
- **Qué representa:** pedidos reales hechos entre 2016 y 2018 en tiendas que venden a través de Olist (Brasil).
- **Archivo principal:** pedidos (`orders`) + productos vendidos (`items`) = **112,650 filas y 14 columnas**
  (numéricas, categóricas y fechas).
- **Archivos secundarios (para el merge):** productos, clientes y reseñas.

## Objetivos del análisis

1. **¿Qué categorías de producto generan más ingreso?**
2. **¿El retraso en la entrega baja la calificación que da el cliente?**
3. **¿Cómo varían las ventas por mes y por estado?**

## 2. Importación de librerías y configuración

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Si pandas lanza un SettingWithCopyWarning, se vuelve error y el notebook se detiene.
# (En pandas 3 esa advertencia ya no existe; el "if" evita fallar en esa versión.)
if hasattr(pd.errors, "SettingWithCopyWarning"):
    warnings.simplefilter("error", pd.errors.SettingWithCopyWarning)

pd.set_option("display.max_columns", None)               # mostrar todas las columnas
pd.set_option("display.float_format", "{:,.2f}".format)  # números con 2 decimales y comas

DATA_DIR = Path("data")  # carpeta donde están los CSV

## 3. Carga e inspección inicial de los datos

- **Principal:** `orders` (pedidos) + `items` (productos de cada pedido), unidos por `order_id`.
  Un pedido puede tener varios productos, así que cada fila de `ventas` es **un producto vendido**.
- **Secundarios:** `products`, `customers` y `reviews`. Se limpian en la sección 4 y se unen en la 5.

In [2]:
# Se cargan los archivos tal como vienen, sin modificarlos
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")

# Tabla principal: pedidos + productos vendidos
ventas = orders.merge(items, on="order_id", how="inner")
print(f"ventas: {ventas.shape[0]:,} filas x {ventas.shape[1]} columnas")

ventas: 112,650 filas x 14 columnas


In [3]:
ventas.head()  # primeras 5 filas

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [4]:
ventas.info()  # tipos de dato y valores no nulos por columna

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       112650 non-null  str    
 1   customer_id                    112650 non-null  str    
 2   order_status                   112650 non-null  str    
 3   order_purchase_timestamp       112650 non-null  str    
 4   order_approved_at              112635 non-null  str    
 5   order_delivered_carrier_date   111456 non-null  str    
 6   order_delivered_customer_date  110196 non-null  str    
 7   order_estimated_delivery_date  112650 non-null  str    
 8   order_item_id                  112650 non-null  int64  
 9   product_id                     112650 non-null  str    
 10  seller_id                      112650 non-null  str    
 11  shipping_limit_date            112650 non-null  str    
 12  price                          112650 non

In [5]:
ventas.describe()  # estadísticas de las columnas numéricas

,order_item_id,price,freight_value
count,"112,650.00","112,650.00","112,650.00"
mean,1.20,120.65,19.99
std,0.71,183.63,15.81
min,1.00,0.85,0.00
25%,1.00,39.90,13.08
50%,1.00,74.99,16.26
75%,1.00,134.90,21.15
max,21.00,"6,735.00",409.68


## 4. Limpieza y preprocesamiento

### 4.1 Tratamiento de valores nulos

In [6]:
# Cuántos nulos hay por columna en la tabla principal
nulos = ventas.isna().sum()
nulos[nulos > 0]

order_approved_at                  15
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
dtype: int64

**Decisiones (una por tipo de nulo):**

1. **Pedido `delivered` sin fecha de entrega → se elimina.** Es un registro inconsistente y no sirve para
   medir retrasos. Son muy pocas filas.
2. **Fecha de aprobación vacía → se imputa con la mediana.** Se calcula cuánto tarda normalmente un pedido en
   aprobarse y se suma a la fecha de compra. Se usa la **mediana** y no la media porque unos pocos pedidos
   tardaron muchísimo y subirían el promedio.
3. **Fecha de entrega vacía en pedidos no entregados → se deja vacía.** Un pedido cancelado o en camino
   todavía no tiene fecha de entrega; inventarla sería falso.
4. **Productos:** categoría vacía → `sin_categoria`; medidas vacías (peso, tamaño) → mediana.

In [7]:
# 1) Pedidos "delivered" sin fecha de entrega -> se eliminan
sin_fecha = (ventas["order_status"] == "delivered") & ventas["order_delivered_customer_date"].isna()
ventas = ventas.loc[~sin_fecha].copy()
print(f"Filas eliminadas: {sin_fecha.sum()}")

# 2) Fecha de aprobación vacía -> fecha de compra + mediana del tiempo de aprobación
compra = pd.to_datetime(ventas["order_purchase_timestamp"])
aprobado = pd.to_datetime(ventas["order_approved_at"])
mediana = (aprobado - compra).median()
ventas["order_approved_at"] = aprobado.fillna(compra + mediana)
print(f"Mediana del tiempo de aprobación: {mediana}")

# 3) Fechas de entrega de pedidos no entregados: se dejan vacías a propósito

# 4) Productos: categoría vacía -> "sin_categoria"; columnas numéricas vacías -> mediana
products["product_category_name"] = products["product_category_name"].fillna("sin_categoria")
numericas = products.select_dtypes("number").columns
products[numericas] = products[numericas].fillna(products[numericas].median())
print(f"Nulos que quedan en products: {products.isna().sum().sum()}")

Filas eliminadas: 8


Mediana del tiempo de aprobación: 0 days 00:21:02
Nulos que quedan en products: 0


### 4.2 Manejo de duplicados

En `reviews` hay pedidos con **más de una reseña**. Si se dejan, al unir las tablas cada venta se repetiría
y el ingreso saldría inflado. Se conserva **la reseña más reciente** de cada pedido.

In [8]:
# Filas idénticas en la tabla principal
print("Filas duplicadas en ventas:", ventas.duplicated().sum())

# Pedidos con más de una reseña
print("Pedidos con reseña repetida:", reviews.duplicated(subset="order_id").sum())

# Ordenar por fecha de respuesta y quedarse con la última reseña de cada pedido
reviews = reviews.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last")
print("Pedidos con reseña repetida después:", reviews.duplicated(subset="order_id").sum())

Filas duplicadas en ventas: 0
Pedidos con reseña repetida: 559
Pedidos con reseña repetida después: 0


### 4.3 Corrección de tipos de datos (casteo)

Las fechas llegaron como **texto**. Se convierten a `datetime` para poder restarlas (calcular retrasos)
y agruparlas por mes.

In [9]:
columnas_fecha = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
]
for columna in columnas_fecha:
    ventas[columna] = pd.to_datetime(ventas[columna])  # texto -> fecha

ventas.dtypes  # comprobar que ahora son datetime64

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
order_item_id                             int64
product_id                                  str
seller_id                                   str
shipping_limit_date              datetime64[us]
price                                   float64
freight_value                           float64
dtype: object